# Classification du jeu Imagenet-1K "samples" avec Alexnet

- Creation : *18/02/2025*

Constat de la performance de classification du modèle.
Utilisation de la définition du modèle et des poids disponibles dans PyTorch.

- [ ] Essayer AlexNet_Weights.IMAGENET1K_V1.transforms

# Module

In [ ]:
import re

import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader
from tqdm import tqdm

from datasets import DATASET_0, DATASET_2, CustomImageDataset, get_label_from_filename
from utils.alexnet_for_deconv import alexnetfordeconv
from imagenet_labels import imagenet1K_names_to_labels, imagenet1K_labels_to_names

# Device

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


# Chargement des données

Création du dataset sur les images

In [ ]:
#geo_transforms = T.Compose([
#    T.Resize(256),
#    T.CenterCrop(224),
#])

#imagenet_mean = DATASET_0["means"]
#imagenet_std = DATASET_0["stds"]
#transforms = T.Compose([
#    T.Resize(256),
#    T.CenterCrop(224),
#    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
#    T.Normalize(mean=imagenet_mean, std=imagenet_std),
#])

transforms = torchvision.models.AlexNet_Weights.IMAGENET1K_V1.transforms()
print(transforms)
get_label = lambda f: get_label_from_filename(f, imagenet1K_names_to_labels)
dataset = CustomImageDataset(DATASET_2["path"], transform=transforms, extension="JPEG", dataloader_mode=True, get_label=get_label)

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [5]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

Test dataloader

In [6]:
batch = next(iter(dataloader))
print(batch[0].size())
print(batch[1].size())

torch.Size([16, 3, 224, 224])
torch.Size([16])


# Chargement du modèle

In [7]:
model_alexnet_deconv = alexnetfordeconv(weights='IMAGENET1K_V1')
model_alexnet_deconv.eval()
model_alexnet_deconv.to(device)

AlexNetForDeconv(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bia

In [ ]:
deconvolution = model_alexnet_deconv(batch[0][0:1].to(device))
print(deconvolution)

# Classification

In [ ]:
for i, batch in tqdm(enumerate, dataloader):
    
